# 01 — Raw data EDA (Torvik, tournament, coaches)

Sanity-check ingested parquets under `data/raw/` and `data/processed/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..").resolve()
TM = ROOT / "data" / "raw" / "torvik" / "timemachine"
FF = ROOT / "data" / "raw" / "torvik" / "four_factors"
PS = ROOT / "data" / "raw" / "torvik" / "player_stats"
PROC = ROOT / "data" / "processed"

## 1. Timemachine pretournament files

Per year: shape, `head(5)`, null counts, min/max for `adjoe`, `adjde`, `barthag`.

In [ ]:
paths = sorted(TM.glob("*_pretournament.parquet"))
flags = []
for p in paths:
    df = pd.read_parquet(p)
    year = p.stem.split("_")[0]
    print("=" * 60, f"\nYEAR {year}  {p.name}  shape={df.shape}")
    print(df.head(5))
    nulls = df.isna().sum()
    print("null counts (top 10):", nulls.nlargest(10).to_dict())
    for col in ("adjoe", "adjde", "barthag"):
        if col in df.columns:
            v = pd.to_numeric(df[col], errors="coerce")
            print(f"  {col}: min={v.min():.4f} max={v.max():.4f}")
    # Heuristic: team column should look like names, not conferences
    bad = df["team"].astype(str).str.match(r"^[A-Z0-9]{2,4}$", na=False) & (df["team"].str.len() <= 4)
    if bad.mean() > 0.3:
        flags.append(f"{year}: possible misaligned `team` column ({bad.mean():.0%} short tokens)")
print("\nFLAGS:", flags)

## 2. Tournament training set

In [ ]:
tt = pd.read_parquet(ROOT / "data" / "raw" / "torvik" / "tournament_training_set.parquet")
print("shape", tt.shape)
print("columns (first 40)", tt.columns[:40].tolist())
print("seasons", sorted(tt["season"].dropna().unique()))
# Winner balance
w1 = (tt["winner"] == tt["team1"]).mean()
print("P(winner is team1)", round(w1, 4), "(expect ~0.5)")

## 3. Four factors — spot-check 2015, 2019, 2023

In [ ]:
for yr in (2015, 2019, 2023):
    p = FF / f"{yr}_four_factors.parquet"
    df = pd.read_parquet(p)
    print(yr, df.shape, df[["team", "efg_pct"]].head(3))

## 4. Player stats — BPM range, min_per, nulls

In [ ]:
for yr in (2015, 2024):
    df = pd.read_parquet(PS / f"{yr}_player_stats.parquet")
    bpm = pd.to_numeric(df["bpm"], errors="coerce")
    mp = pd.to_numeric(df["min_per"], errors="coerce")
    print(yr, "BPM", bpm.min(), bpm.max(), "min_per sum/team sample", df.groupby("team")["min_per"].sum().head(3).tolist())
    print(" null rate", df.isna().mean().sort_values(ascending=False).head(5).to_dict())

## 5. Coach store — distribution + cumulative check

In [ ]:
cs = pd.read_parquet(PROC / "coach_store.parquet")
print("shape", cs.shape)
print(cs["coach_tourn_appearances"].describe())
# 2026 season row should use only prior seasons — flag if season_year max looks wrong
print("season_year range", cs["season_year"].min(), cs["season_year"].max())
print("sample 2025 rows", cs[cs["season_year"] == 2025].head(3))

---

## Summary (findings)

- **Timemachine / four_factors `team` column:** Early-year parquets can show **misaligned columns** (team vs conf); `build_static.py` merges four-factors to timemachine on **`(year, rank)`** row alignment instead of team name.
- **Tournament set:** `t1wp`/`t2wp` are binary favorite flags in this extract; upset analytics use **efficiency net** (`t1adjo-t1adjd` vs `t2adjo-t2adjd`).
- **Coach store:** `season_year` may include bad parses (e.g. 2030s) from malformed season strings — filter or fix in a future ingest pass.
- **Player stats:** `min_per` sums to ~500 per team (Torvik convention); BPM ranges are plausible for D1.

_Run all cells above, then paste any new flags below._
